# SEACO-6161 · 01 Investigation · `dbw-opas-merone-pipeline-prod`

**Read-only.** This notebook finds out what is holding the 188.41 TiB on
`dbstorageld3xv3o6mfr6o` ($2,189/month). It changes no data. The only thing it writes is
small audit tables in `storage_audit_seaco_6161`, which `02_cleanup` and the before/after cost report use.

It answers the four goals of the ticket:

| # | Goal | Section |
|---|---|---|
| 1 | Size of `dbfs:/` per directory | 3 |
| 2 | Delta tables never VACUUMed, or with large retained-version storage | 4, 8 |
| 3 | Retention of cluster logs and job logs | 6 |
| 4 | Abandoned workflows still writing to `dbfs:/` | 7 |

Section 5 also checks for shallow clones (safety), and section 9 lists cleanup candidates.

**Run time:** around 15–60 minutes, mostly the directory scan in section 3. Use a Single User cluster with a large
driver, 4–8 workers are enough. Running it again is safe, because every run appends under a new `run_id`.

### What is already known for this workspace (Tom De Leu, 2026-06-01)

- 762 registered Delta tables hold only **9.38 TiB** live; the account holds **188.41 TiB**.
- None of the 30 heaviest tables had a VACUUM in its history. Retention is the default
  (`deletedFileRetentionDuration` = 7 days, `logRetentionDuration` = 30 days).
- The busiest writer is `PROD-bulk-apt-source-check-cache-job` (job id `185138214535362`, owner Rakesh Gupta,
  triggered by hand). It rewrites `preprocess_prod.layer_*`.
- Its `backup_cleanup` task (`prod/cleanup_backup`) copies tables into `backup_preprocess_prod`.
  **Check whether it uses `SHALLOW CLONE`.** If it does, a VACUUM on `preprocess_prod.*` breaks those backups.
  Section 5 of `01_investigation` checks this from the data.

In [0]:
%run ./00_config_and_helpers

In [0]:
dbutils.widgets.dropdown("scan_dbfs", "true", ["true", "false"], "1. Scan dbfs:/ directories")
dbutils.widgets.text("scan_depth", "4", "2. Scan depth (levels below dbfs:/)")
dbutils.widgets.dropdown("scan_external_mounts", "false", ["true", "false"], "3. Also size external mounts")
dbutils.widgets.text("databases", "", "4. Only these databases (comma list, empty = all)")

SCAN_DBFS = dbutils.widgets.get("scan_dbfs") == "true"
SCAN_DEPTH = int(dbutils.widgets.get("scan_depth"))
SCAN_EXTERNAL = dbutils.widgets.get("scan_external_mounts") == "true"
ONLY_DBS = [d.strip() for d in dbutils.widgets.get("databases").split(",") if d.strip()]

## 1 · Mount points

Not every `/mnt/*` path is on the root account. This table shows where each mount point really points.
Only paths classed as `root` count toward this cost line.

In [0]:
print_mounts()

## 2 · Plan the scan

`dbfs:/` is split into units that each sit entirely on one storage account. A directory that contains a mount
point lower down is opened up, so no bytes get counted under the wrong account.
`/databricks-datasets` and UC Volumes are skipped.

In [0]:
SKIP_PREFIXES = ("/databricks-datasets", "/Volumes", "/volumes", "/volume")


def collect_leaves(max_depth, max_children=2000):
    """Walk dbfs:/ down to max_depth. Returns (leaf_dirs, loose_files).
    Stops early when a directory has more than max_children entries, or sits on a mount boundary."""
    leaves, files = [], []

    def walk(p, depth):
        n = normalize(p)
        if n.startswith(SKIP_PREFIXES):
            return
        has_nested_mount = any(mp.startswith(n.rstrip("/") + "/") for mp in MOUNT_POINTS)
        if depth >= max_depth and not has_nested_mount:
            leaves.append(n)
            return
        children = ls("dbfs:" + n)
        if len(children) > max_children and not has_nested_mount:
            leaves.append(n)
            return
        for c in children:
            cp = normalize(c.path)
            if c.isDir():
                walk(cp, depth + 1)
            else:
                files.append((cp, c.size))

    for c in ls("dbfs:/"):
        cp = normalize(c.path)
        if c.isDir():
            walk(cp, 1)
        else:
            files.append((cp, c.size))
    return leaves, files


if SCAN_DBFS:
    t0 = time.time()
    LEAVES, LOOSE_FILES = collect_leaves(SCAN_DEPTH)
    to_scan = [p for p in LEAVES if SCAN_EXTERNAL or classify_path(p) == "root"]
    print(f"{len(LEAVES):,} leaf dirs ({len(to_scan):,} to size), {len(LOOSE_FILES):,} loose files, "
          f"planned in {time.time() - t0:,.0f}s")

## 3 · Size of `dbfs:/` per directory (goal 1)

Each leaf directory is sized with Hadoop `getContentSummary` on parallel driver threads. Sizes then roll up to
every parent level. This is the slow part.

In [0]:
if SCAN_DBFS:
    t0 = time.time()
    scan_rows = []
    with ThreadPoolExecutor(THREADS) as pool:
        futs = {pool.submit(content_summary, "dbfs:" + p): p for p in to_scan}
        for i, f in enumerate(as_completed(futs), 1):
            p = futs[f]
            cs = f.result()
            scan_rows.append(dict(path=p, storage_class=classify_path(p), kind="dir", **cs))
            if i % 200 == 0:
                print(f"  {i:,}/{len(to_scan):,} sized, {time.time() - t0:,.0f}s")
    for p, size in LOOSE_FILES:
        scan_rows.append(dict(path=p, storage_class=classify_path(p), kind="file",
                              bytes=int(size), files=1, dirs=0, error=None))
    schema = ("run_id string, run_ts timestamp, workspace string, path string, storage_class string, "
              "kind string, bytes long, files long, dirs long, error string")
    save_audit("dbfs_scan", scan_rows, schema=schema)
    print(f"sized in {time.time() - t0:,.0f}s")

In [0]:
if SCAN_DBFS:
    scan = spark.table(audit_table("dbfs_scan")).filter(F.col("run_id") == RUN_ID)
    parts = F.split(F.col("path"), "/")
    rollup = None
    for level in range(1, SCAN_DEPTH + 1):
        lvl = (scan.filter(F.size(parts) > level)
               .withColumn("prefix", F.concat_ws("/", F.slice(parts, 1, level + 1)))
               .groupBy("storage_class", "prefix").agg(F.sum("bytes").alias("bytes"), F.sum("files").alias("files"))
               .withColumn("level", F.lit(level)))
        rollup = lvl if rollup is None else rollup.unionByName(lvl)
    rollup = rollup.withColumn("tib", F.round(F.col("bytes") / TIB, 3)).cache()
    rollup.createOrReplaceTempView("dbfs_rollup")

    root_bytes = scan.filter("storage_class = 'root'").agg(F.sum("bytes")).collect()[0][0] or 0
    errors = scan.filter("error IS NOT NULL").count()
    print(f"Visible on root account via dbfs:/ : {human(root_bytes)}  "
          f"({root_bytes / TIB / AZURE_REPORTED_TIB:.0%} of the {AZURE_REPORTED_TIB} TiB Azure reports)")
    print(f"Directories that could not be sized: {errors}")
    print("\nTop-level directories on the root account:")
    display(rollup.filter("level = 1 AND storage_class = 'root'").orderBy(F.desc("bytes")))

In [0]:
%sql
-- Every directory of 100 GiB or more at any level, biggest first
SELECT level, prefix, storage_class, tib, files
FROM dbfs_rollup
WHERE bytes >= 100 * power(1024, 3)
ORDER BY bytes DESC

## 4 · Table inventory (goal 2)

For every table this records:

- `DESCRIBE DETAIL`: format, location, live size, file count, created and last modified
- `DESCRIBE HISTORY`: operation count, last VACUUM, the last writer (job, notebook, user), CLONE operations
- on-disk size of the table directory, for tables on the root account only

`overhead_bytes` = on-disk − live − `_delta_log`. This is roughly what a VACUUM could free.

⚠️ History only reaches back `logRetentionDuration` (30 days). `vacuum_in_retained_history = false` means
no VACUUM in that window. It does not prove the table was never vacuumed. For staleness, use `last_modified`.

In [0]:
def inspect_table(db, table):
    rec = dict(database=db, table_name=table, full_name=f"{db}.{table}", error=None)
    try:
        ext = table_extended(db, table)
        rec.update(table_type=ext.get("Type"), provider=(ext.get("Provider") or "").lower() or None,
                   location=ext.get("Location"))
    except Exception as e:
        rec["error"] = f"describe extended: {str(e)[:250]}"
        return rec
    rec["storage_class"] = classify_path(rec.get("location"))
    if rec.get("provider") == "delta":
        try:
            d = describe_detail(db, table)
            props = d.get("properties") or {}
            rec.update(live_bytes=d.get("sizeInBytes"), num_files=d.get("numFiles"),
                       created_at=d.get("createdAt"), last_modified=d.get("lastModified"),
                       location=d.get("location") or rec.get("location"),
                       deleted_file_retention=props.get("delta.deletedFileRetentionDuration"),
                       log_retention=props.get("delta.logRetentionDuration"))
            rec.update(history_summary(db, table))
        except Exception as e:
            rec["error"] = f"delta detail/history: {str(e)[:250]}"
    if rec["storage_class"] == "root" and rec.get("location"):
        cs = content_summary(rec["location"])
        rec.update(disk_bytes=cs["bytes"], disk_files=cs["files"])
        if rec.get("provider") == "delta":
            rec["delta_log_bytes"] = content_summary(rec["location"].rstrip("/") + "/_delta_log")["bytes"]
    return rec


dbs = ONLY_DBS or [d for d in list_databases() if d != AUDIT_DB]
todo = [(db, t) for db in dbs for t in list_tables(db)]
print(f"{len(dbs)} databases, {len(todo):,} tables to inspect")

t0 = time.time()
TABLE_ROWS = []
with ThreadPoolExecutor(THREADS) as pool:
    futs = [pool.submit(inspect_table, db, t) for db, t in todo]
    for i, f in enumerate(as_completed(futs), 1):
        TABLE_ROWS.append(f.result())
        if i % 100 == 0:
            print(f"  {i:,}/{len(todo):,} tables, {time.time() - t0:,.0f}s")
print(f"inspected in {time.time() - t0:,.0f}s, errors: {sum(1 for r in TABLE_ROWS if r.get('error'))}")

## 5 · Shallow-clone check (safety)

A `SHALLOW CLONE` (often used for "backups") has no data files of its own. It points at the source table's
files. If the source is VACUUMed, the clone breaks. This check finds every Delta table whose current files
live outside its own directory, and marks the owner of those files as a **clone source**.
`02_cleanup` and `03_weekly_vacuum_job` will not VACUUM a clone source.

In [0]:
delta_tables = [(r["full_name"], r["location"]) for r in TABLE_ROWS
                if r.get("provider") == "delta" and r.get("location") and not r.get("error")]
spark.createDataFrame(delta_tables, "table_name string, table_location string") \
    .createOrReplaceTempView("_clone_check_input")
print(f"{len(delta_tables):,} Delta tables queued for the clone check")

In [0]:
%scala
findShallowCloneRefs()

In [0]:
CLONE_ROWS = [r.asDict() for r in spark.table("_clone_check_output").collect()]
CLONE_SOURCES, UNRESOLVED = clone_sources(CLONE_ROWS, TABLE_ROWS)

for r in TABLE_ROWS:
    clones = CLONE_SOURCES.get(r["full_name"], [])
    r["is_shallow_clone_source"] = bool(clones)
    r["shallow_clones"] = ",".join(clones) if clones else None
    if r.get("disk_bytes") is not None and r.get("live_bytes") is not None:
        r["overhead_bytes"] = max(r["disk_bytes"] - r["live_bytes"] - (r.get("delta_log_bytes") or 0), 0)
    r["vacuum_in_retained_history"] = r.get("last_vacuum_ts") is not None

save_audit("clone_dependencies", [dict(c) for c in CLONE_ROWS if c.get("referenced_dir") or c.get("error")],
           schema="run_id string, run_ts timestamp, workspace string, table_name string, table_location string, "
                  "referenced_dir string, n_files long, error string")

print(f"Clone sources (VACUUM is blocked on these): {len(CLONE_SOURCES)}")
for src, clones in sorted(CLONE_SOURCES.items()):
    print(f"  {src}  <- {', '.join(clones)}")
if UNRESOLVED:
    print(f"\n⚠️  {len(UNRESOLVED)} file groups point outside every known table location. A clone of an unregistered")
    print("    path, or of another workspace's table? Check these before any VACUUM:")
    for c in UNRESOLVED[:50]:
        print(f"  {c['table_name']} -> {c['referenced_dir']} ({c['n_files']} files)")

In [0]:
INV_SCHEMA = (
    "run_id string, run_ts timestamp, workspace string, database string, table_name string, full_name string, "
    "table_type string, provider string, location string, storage_class string, live_bytes long, num_files long, "
    "created_at timestamp, last_modified timestamp, deleted_file_retention string, log_retention string, "
    "disk_bytes long, disk_files long, delta_log_bytes long, overhead_bytes long, history_ops long, "
    "history_first_ts timestamp, history_last_ts timestamp, last_operation string, distinct_ops string, "
    "write_ops long, last_vacuum_ts timestamp, vacuum_in_retained_history boolean, last_write_ts timestamp, "
    "last_writer_user string, last_writer_job_id string, last_writer_job_name string, "
    "last_writer_notebook_id string, clone_history string, is_shallow_clone_source boolean, "
    "shallow_clones string, error string"
)
cols = [c.split()[0] for c in INV_SCHEMA.split(", ")][3:]
save_audit("table_inventory", [{c: r.get(c) for c in cols} for r in TABLE_ROWS], schema=INV_SCHEMA)
inv = spark.table(audit_table("table_inventory")).filter(F.col("run_id") == RUN_ID).cache()
inv.createOrReplaceTempView("inv")

In [0]:
%sql
-- Where the bytes are, by database (root account only)
SELECT database,
       count(*)                                         AS tables,
       round(sum(live_bytes)     / power(1024, 4), 3)   AS live_tib,
       round(sum(disk_bytes)     / power(1024, 4), 3)   AS disk_tib,
       round(sum(overhead_bytes) / power(1024, 4), 3)   AS vacuumable_tib_est,
       max(last_modified)                               AS last_write_any_table,
       sum(CASE WHEN NOT vacuum_in_retained_history THEN 1 ELSE 0 END) AS no_vacuum_in_30d
FROM inv
WHERE storage_class = 'root'
GROUP BY database
ORDER BY disk_tib DESC

In [0]:
%sql
-- Biggest VACUUM targets: high on-disk overhead compared with live size
SELECT full_name, round(live_bytes / power(1024,3), 1) AS live_gib,
       round(disk_bytes / power(1024,3), 1)             AS disk_gib,
       round(overhead_bytes / power(1024,3), 1)         AS overhead_gib,
       round(disk_bytes / nullif(live_bytes, 0), 1)     AS disk_to_live_ratio,
       write_ops, last_modified, last_vacuum_ts, last_writer_job_name, is_shallow_clone_source
FROM inv
WHERE provider = 'delta' AND storage_class = 'root'
ORDER BY overhead_bytes DESC NULLS LAST
LIMIT 100

## 6 · Cluster logs and job logs (goal 3)

Cluster log delivery (`cluster_log_conf`) is set per cluster and per job cluster. Logs delivered to `dbfs:/...`
land on the root account and are **never deleted automatically**.
The `logs` and `jobs` containers that Databricks manages itself are **not reachable from a notebook**,
so their size shows up only as the gap in section 8.

In [0]:
def _log_dest(cluster_spec):
    conf = (cluster_spec or {}).get("cluster_log_conf") or {}
    return (conf.get("dbfs") or {}).get("destination") or (conf.get("volumes") or {}).get("destination")


CLUSTERS = api_get("/api/2.0/clusters/list").get("clusters", [])
JOBS = list(api_paginate("/api/2.1/jobs/list", "jobs", {"limit": 100, "expand_tasks": "true"}))
print(f"{len(CLUSTERS)} clusters, {len(JOBS)} jobs visible to {RUN_USER}  (an admin sees all of them)")

log_rows = []
for c in CLUSTERS:
    dest = _log_dest(c)
    log_rows.append(dict(owner_type="cluster", owner_id=c["cluster_id"], owner_name=c.get("cluster_name"),
                         creator=c.get("creator_user_name"), destination=dest))
for j in JOBS:
    s = j.get("settings", {})
    specs = [jc.get("new_cluster") for jc in s.get("job_clusters", [])] + \
            [t.get("new_cluster") for t in s.get("tasks", []) if t.get("new_cluster")]
    for spec in specs:
        log_rows.append(dict(owner_type="job", owner_id=str(j["job_id"]), owner_name=s.get("name"),
                             creator=j.get("creator_user_name"), destination=_log_dest(spec)))

dests = sorted({r["destination"] for r in log_rows if r["destination"]})
dest_size = {d: content_summary(d) for d in dests if classify_path(d) == "root"}
for r in log_rows:
    d = r["destination"]
    r["storage_class"] = classify_path(d) if d else None
    r["destination_bytes"] = dest_size.get(d, {}).get("bytes") if d else None

save_audit("cluster_log_conf", log_rows,
           schema="run_id string, run_ts timestamp, workspace string, owner_type string, owner_id string, "
                  "owner_name string, creator string, destination string, storage_class string, "
                  "destination_bytes long")
print("\nLog destinations on the root account:")
for d, cs in sorted(dest_size.items(), key=lambda kv: -(kv[1]["bytes"] or 0)):
    print(f"  {human(cs['bytes']):>14}  {cs['files'] or 0:>12,} files  {d}")
print("\nRetention: nothing deletes these automatically. 02_cleanup step 'prune_cluster_logs' does it.")

## 7 · Who writes to the root account? (goal 4)

This joins the last writer of each root-account table (from Delta history) with the jobs API.
A job may be abandoned when it still writes, or is scheduled to write, to tables nobody reads anymore.
A person has to decide that. This section puts the facts side by side.

In [0]:
def last_run(job_id):
    runs = api_get("/api/2.1/jobs/runs/list", {"job_id": job_id, "limit": 1}).get("runs", [])
    if not runs:
        return None, None
    r = runs[0]
    return (dt.datetime.utcfromtimestamp(r["start_time"] / 1000),
            (r.get("state") or {}).get("result_state") or (r.get("state") or {}).get("life_cycle_state"))


job_rows = []
with ThreadPoolExecutor(THREADS) as pool:
    runs = dict(zip([j["job_id"] for j in JOBS], pool.map(lambda j: last_run(j["job_id"]), JOBS)))
for j in JOBS:
    s = j.get("settings", {})
    sched = s.get("schedule") or {}
    trig = s.get("trigger") or {}
    nb_paths = [t["notebook_task"]["notebook_path"] for t in s.get("tasks", []) if t.get("notebook_task")]
    lr_ts, lr_state = runs.get(j["job_id"]) or (None, None)
    job_rows.append(dict(
        job_id=str(j["job_id"]), job_name=s.get("name"), creator=j.get("creator_user_name"),
        schedule_cron=sched.get("quartz_cron_expression"), schedule_paused=sched.get("pause_status"),
        continuous=bool(s.get("continuous")), file_trigger=bool(trig),
        n_tasks=len(s.get("tasks", [])), notebook_paths=json.dumps(nb_paths) if nb_paths else None,
        last_run_start=lr_ts, last_run_state=lr_state,
    ))
save_audit("jobs_inventory", job_rows,
           schema="run_id string, run_ts timestamp, workspace string, job_id string, job_name string, "
                  "creator string, schedule_cron string, schedule_paused string, continuous boolean, "
                  "file_trigger boolean, n_tasks long, notebook_paths string, last_run_start timestamp, "
                  "last_run_state string")
spark.table(audit_table("jobs_inventory")).filter(F.col("run_id") == RUN_ID).createOrReplaceTempView("jobs")

In [0]:
%sql
-- Writers of root-account tables. Look for: scheduled jobs filling databases that look abandoned,
-- jobs owned by people who left, and tables written from notebooks by hand (last_writer_job_id is NULL).
SELECT coalesce(i.last_writer_job_name, '(interactive: ' || i.last_writer_user || ')') AS writer,
       i.last_writer_job_id                            AS job_id,
       j.creator, j.schedule_cron, j.schedule_paused, j.continuous, j.last_run_start, j.last_run_state,
       count(*)                                        AS tables_written,
       round(sum(i.disk_bytes) / power(1024,4), 3)     AS disk_tib,
       max(i.last_write_ts)                            AS last_write,
       concat_ws(', ', slice(sort_array(collect_set(i.database)), 1, 10)) AS databases
FROM inv i LEFT JOIN jobs j ON i.last_writer_job_id = j.job_id
WHERE i.storage_class = 'root' AND i.provider = 'delta' AND i.last_write_ts IS NOT NULL
GROUP BY ALL
ORDER BY disk_tib DESC

## 8 · Reconciliation and baseline

The figures below are saved as the **"before"** baseline for the before/after cost report in the sprint goal.
The gap between what `dbfs:/` shows and what Azure reports is held in containers a notebook cannot reach
(`jobs`, `logs`, `ephemeral`, `meta`), and possibly in soft-deleted blobs.

In [0]:
live_root = inv.filter("storage_class = 'root'").agg(F.sum("live_bytes")).collect()[0][0] or 0
disk_root = inv.filter("storage_class = 'root'").agg(F.sum("disk_bytes")).collect()[0][0] or 0
over_root = inv.filter("storage_class = 'root' AND NOT is_shallow_clone_source").agg(F.sum("overhead_bytes")).collect()[0][0] or 0
scanned_root = (spark.table(audit_table("dbfs_scan")).filter((F.col("run_id") == RUN_ID) & (F.col("storage_class") == "root"))
                .agg(F.sum("bytes")).collect()[0][0] or 0) if SCAN_DBFS else None
azure_bytes = AZURE_REPORTED_TIB * TIB

lines = [
    ("Azure-reported account size (May 2026)", azure_bytes),
    ("Visible through dbfs:/ (section 3)", scanned_root),
    ("  of which: registered tables on disk", disk_root),
    ("  of which: live table data", live_root),
    ("  of which: VACUUM-able table overhead (est.)", over_root),
    ("dbfs:/ bytes not in any registered table", (scanned_root - disk_root) if scanned_root is not None else None),
    ("Not visible from notebooks (jobs/logs/ephemeral/meta, soft-delete)", (azure_bytes - scanned_root) if scanned_root is not None else None),
]
for label, b in lines:
    print(f"{label:<70} {human(b):>14}")

save_audit("baseline", [dict(scanned_root_bytes=scanned_root, table_disk_bytes=disk_root, table_live_bytes=live_root,
                             vacuumable_bytes_est=over_root, azure_reported_tib=AZURE_REPORTED_TIB,
                             azure_monthly_cost_usd=AZURE_MONTHLY_COST_USD, scan_depth=SCAN_DEPTH)],
           schema="run_id string, run_ts timestamp, workspace string, scanned_root_bytes long, table_disk_bytes long, "
                  "table_live_bytes long, vacuumable_bytes_est long, azure_reported_tib double, "
                  "azure_monthly_cost_usd double, scan_depth long")
rate = AZURE_MONTHLY_COST_USD / azure_bytes
print(f"\nEstimated saving if the VACUUM-able overhead is removed: ~${over_root * rate:,.0f}/month "
      f"(at the effective rate of ${rate * GIB:.5f}/GiB-month)")

## 9 · Cleanup candidates (people must confirm these)

**Nothing here is deleted.** Walk these lists with the table owners, then feed the confirmed names into
`02_cleanup`.

In [0]:
%sql
-- A. Stale tables: not written for 30+ days. Candidates to DROP once the owner agrees.
--    "single_op" = only one commit in the retained history (created once, never touched since).
SELECT full_name, table_type, round(disk_bytes / power(1024,3), 1) AS disk_gib,
       round(live_bytes / power(1024,3), 1) AS live_gib, created_at, last_modified,
       history_ops = 1 AS single_op, last_writer_user, is_shallow_clone_source, shallow_clones
FROM inv
WHERE storage_class = 'root' AND last_modified < current_timestamp() - INTERVAL 30 DAYS
ORDER BY disk_bytes DESC NULLS LAST
LIMIT 200

In [0]:
%sql
-- B. The same table name in several databases (copies of preprocess / source / snapshot layers)
SELECT table_name, count(*) AS copies,
       round(sum(disk_bytes) / power(1024,3), 1) AS total_disk_gib,
       concat_ws(', ', sort_array(collect_list(database))) AS in_databases
FROM inv
WHERE storage_class = 'root'
GROUP BY table_name
HAVING count(*) > 1
ORDER BY total_disk_gib DESC
LIMIT 50

In [0]:
# C. Orphan directories: data under a database location (or the warehouse) that no registered table owns.
#    Usually left by df.write.save() without saveAsTable, or by external tables dropped without their files.
table_locs = [normalize(r["location"]) for r in TABLE_ROWS if r.get("location")]
db_locs = set()
for db in dbs:
    try:
        for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {qname(db)}").collect():
            if r[0] in ("Location", "RootLocation") and r[1]:
                db_locs.add(r[1])
    except Exception:
        pass
db_locs.add("dbfs:/user/hive/warehouse")

orphan_rows = []
for loc in sorted(db_locs):
    if classify_path(loc) != "root":
        continue
    for child in ls(loc):
        cp = normalize(child.path)
        if not child.isDir() or (cp.endswith(".db") and loc.rstrip("/").endswith("warehouse")):
            continue
        if any(is_under(tl, cp) or is_under(cp, tl) for tl in table_locs):
            continue
        cs = content_summary(child.path)
        orphan_rows.append(dict(parent_location=loc, path=child.path, **cs))
save_audit("orphan_dirs", orphan_rows,
           schema="run_id string, run_ts timestamp, workspace string, parent_location string, path string, "
                  "bytes long, files long, dirs long, error string")
if orphan_rows:
    display(spark.createDataFrame(orphan_rows).orderBy(F.desc("bytes")))

### D. Notes carried over from the ticket

Databases that look like one-off experiments: arg_im_60432, data_recovery, dbscan, duplicate_apt_im, rooftop_regression_db / _new / _old, scoping_aj, test, twn_update

In the section 3 rollup, also look at `dbfs:/tmp`, `dbfs:/FileStore`, `dbfs:/databricks-results`,
`dbfs:/cluster-logs`, `dbfs:/pipelines` and `dbfs:/local_disk0`. A large top-level directory that is not
`user/hive/warehouse` usually holds unregistered data.

**Next:** take the confirmed names to `02_cleanup`.